# Day 2 まとめ: Databricks × Dify 連携

---

## 1. 連携パターン一覧

| # | パターン | 方向 | 検証状況 | Dify連携 | 推奨場面 |
|---|---------|------|---------|----------|--------|
| ① | LLMモデル連携 | Dify→DB | ✅ 検証済み | ◎ | LLMの一元管理・コスト制御 |
| ② | HTTP API連携 | Dify→DB | ✅ 検証済み | ◎〜○ | UC関数・Vector Search・Genie・KA/MAS/Agent |
| ③ | MCP連携 | Dify→DB | ✅ 検証済み | ◎ | 標準プロトコルでのツール連携 |
| ④ | RAG（External Knowledge） | Dify→DB | ⚠️ 紹介のみ | △ | Dify標準ナレッジ機能の活用（中間API必要） |
| ⑤ | Databricksオーケストレーター | DB→Dify | ⚠️ 紹介のみ | ○ | 大規模バッチAI処理 |
| ⑥ | 観測性/MLOps | Dify→DB | ✅ 検証済み | ○ | 品質管理・トレース・AI Judge |

### 補足
- **④**: Databricks Vector Search APIとDifyのExternal Knowledge APIのフォーマットが異なるため、変換用の中間APIサーバーが必要
- **⑤**: ローカルDockerのDifyにはDatabricksからアクセスできないため未検証（Dify Cloud/パブリックデプロイが必要）
- **⑥**: Dify v1.10.1で追加されたDatabricksネイティブプロバイダーを使用。一部制約・バグあり（後述）

## 2. 今回の検証で分かったこと

### LLMモデル連携（Pattern ①）
- GPT系モデル（`databricks-gpt-5-2`）はDifyからそのまま利用可能（パッチ不要）
- Claude/Gemini系はDifyプラグインが`user`パラメータを送るため、プラグインのパッチが必要
- Dify公式Anthropicプラグインは認証ヘッダーが`x-api-key`固定のため、Databricksには接続不可

### HTTP API連携（Pattern ②）
- DatabricksのREST API（UC Functions, Vector Search, Genie, KA/MAS/Agent）は全てDifyのHTTPノードから呼び出し可能
- UC Functions / Vector Searchは**同期API**で最も相性が良い
- Genieは**非同期API**（ポーリング必要）。DifyのCodeノード内でループ実装が必要
- KA/MAS/Agentは**ResponsesAgent形式**（`input`/`output`）。LLMモデルとしては接続不可、HTTPツールとして利用
- `output`配列に`function_call`/`function_call_output`/`message`が混在。最終回答は`type=message`のみ

### MCP連携（Pattern ③）
- Databricks Managed MCP（Streamable HTTP）でUC FunctionsとVector Searchをツールとして公開
- MCPの実体はJSON-RPC over HTTP（`initialize` → `tools/list` → `tools/call`）
- DifyからはMCP SSEプラグイン（`junjiem/mcp_sse`）で接続
- Vector Search MCPの引数は`query_text`ではなく`query`（REST APIと異なる）

### 認証・権限管理
- 全APIパターン共通で`Authorization: Bearer <PAT>`で認証
- DifyアプリにはPATが固定で埋め込まれるため、**ユーザー単位のアクセス制御は不可**
- 部門別制御 → Dify App × SP分離 / ユーザー単位 → Databricks Apps（OAuth U2M）
- Dify Workflowの**環境変数（Secret）** を使えばPATをDSLエクスポートから分離可能

### 観測性/MLOps連携（Pattern ⑥）
- Dify v1.10.1でDatabricks専用トレーシングプロバイダーが追加（MLflowプロバイダーと同時）
- Agent/Chatbot/Workflowの全アプリタイプでMLflow Experimentにトレース送信可能
- **Experiment IDはDifyワークスペース全体で1つ**（アプリごとの分離不可）
- トレースに**アプリ名が含まれない**（Agent/Chatbot）。スパン名で種類を識別
- MCP経由のVector SearchはTOOLスパンになり、`RetrievalGroundedness`スコアラーが使えない
- **Dify v1.13.0バグ**: External Knowledge + トレーシングの併用でNullPointerクラッシュ
- MLflow 3のAI Judge（Safety, Guidelines）による品質自動評価は動作確認済み

## 3. Ricohへの推奨アーキテクチャ

### 現在の課題
```
Dify App × N本（乱立）
  ├── 各アプリにモデル設定が重複
  ├── ツール・ナレッジが散在し再利用できない
  └── 誰が何を作ったか把握できない
```

### 目標アーキテクチャ
```
Databricks（ガバナンス基盤）
┌──────────────────────────────────┐
│ モデル    → AI Gateway（コスト制御）     │
│ ツール    → UC Functions（権限管理）     │
│ ナレッジ  → Vector Search（一元管理）    │
│ 観測性    → MLflow Tracing（品質評価）   │
└──────────────┬───────────────────┘
               │ API / MCP
        ┌──────┼──────┐
       Dify   Dify   Databricks Apps
      (軽量)  (軽量)  (OAuth U2M)
       ↑ UIとプロンプトだけ
```

### 管理対象の移行

| 管理対象 | 現状（Dify内） | 目標（Databricks） |
|---------|-------------|------------------|
| LLMモデル | 各アプリに個別設定 | AI Gatewayで一元管理・コスト制御 |
| ツール | 各アプリに個別実装 | UC Functionsで共有・権限管理 |
| ナレッジ | 各アプリに個別ナレッジ | Vector Searchで一元管理 |
| 品質管理 | なし | MLflow Tracing + AI Judge |
| DSLファイル | モデル+ツール+ナレッジ全部入り | UIとプロンプトだけの薄い層 |

> **核心メッセージ**: DSLファイルを管理するのではなく、  
> DSLの中身（部品）をDatabricksに引き上げるのが本質。  
> アプリが乱立しても、部品はDatabricks側で統制できるので影響が最小化される。

## 4. まとめ

| Ricohの課題 | Databricksの解決策 | 検証結果 |
|------------|-------------------|--------|
| Difyアプリの乱立 | UC で部品を一元管理、Difyは薄いUI層 | ✅ MCP/APIで部品共有を実証 |
| ガバナンスの欠如 | UC権限 + 監査ログ + リネージ | ✅ PAT+UC GRANTで権限制御を確認 |
| ツール・ナレッジの散在 | UC Functions + Vector Search | ✅ MCP/HTTP APIで外部公開を実証 |
| 品質管理の困難さ | MLflow Tracing + AI Judge | ✅ トレース送信+自動評価を実証 |
| モデルコスト管理 | AI Gateway | ✅ GPT-5.2をDify経由で利用を実証 |

### Day1 × Day2 の関係

| | Day1（コードAgent） | Day2（Dify連携） |
|--|-------------------|----------------|
| 開発手法 | LangGraph + Python | ノーコード/ローコード |
| デプロイ | Model Serving Endpoint | Dify Docker |
| 共通部品 | UC Functions, Vector Search | **← 同じ部品を共有** |
| 公開方法 | REST API (ResponsesAgent) | MCP / HTTP API |

> Day1とDay2は対立ではなく、**部品をDatabricksで共有し、UIを使い分ける**のが正しいアーキテクチャ。

## 5. 次のステップ（PoC計画）

### Phase 1: Quick Win（1-2ヶ月）— すぐにPoCできる

| 施策 | 内容 | 今回の検証で実証済み |
|------|------|------------------|
| **① LLMモデル連携** | AI Gateway経由でGPT-5.2をDifyから利用 | ✅ |
| **③ MCP連携** | UC Functions + Vector SearchをMCPで公開 | ✅ |
| **⑥ トレーシング** | DatabricksプロバイダーでMLflowにトレース送信 | ✅ |

**具体的なPoC内容**:
1. Ricohの実業務データでUC Functions（顧客検索、注文履歴等）を作成
2. 製品ドキュメントでVector Search Indexを構築
3. MCP経由でDify Agentに接続 → 社内サポートBotとして運用開始
4. MLflowトレーシングで品質モニタリング

### Phase 2: Deep Integration（2-4ヶ月）

| 施策 | 内容 |
|------|------|
| **② HTTP API拡張** | Genie連携（自然言語SQL）、KA/MAS呼び出し |
| **認証強化** | 専用SP作成 + UC権限の最小化 |
| **AI Judge運用** | 品質評価の自動化（Workflow定期実行） |

### Phase 3: Full Governance（4-6ヶ月）

| 施策 | 内容 |
|------|------|
| **⑤ バッチ処理** | Databricks WorkflowからDify APIでバッチAI処理 |
| **④ External Knowledge** | 中間API構築でDify標準ナレッジ機能を活用 |
| **Databricks Apps** | ユーザー単位の権限制御が必要な場面でOAuth U2Mアプリ |

---

**ハンズオンお疲れ様でした！**